# 002 Models And Messages

这是 LangChain 学习线的第二份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/overview
- https://docs.langchain.com/oss/python/langchain/install
- https://docs.langchain.com/oss/python/langchain/agents

学习目标：

1. 学会 LangChain 的 message 类型
2. 理解 `system` / `user` / `assistant` / `tool` 的上下文结构
3. 理解模型调用前 messages 怎么组织
4. 对比本仓库的 `history_snapshot` 和 `completed_steps`
5. 为后续 tool calling 和 structured output 打底

---

## 1. 先记住一句话

LangChain 里，模型不是直接接收一段裸文本，而是接收一组 messages。

这些 messages 通常包含：

- `system`：系统规则
- `user`：用户输入
- `assistant`：模型上一轮输出
- `tool`：工具返回结果

这和我们当前的 Harness runtime 很像：

- `history_snapshot` 对应多轮 messages 的历史
- `completed_steps` 对应 agent 执行过程中的结构化观察
- `plan` 对应模型下一步意图

这一讲先不讲 agent loop，只看消息层。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 加载项目环境变量

这一讲不强制调用真实模型，但如果你想后面跑 `ChatOpenAI.invoke(...)`，还是要先从项目根目录加载 `.env`。

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)

Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 3. 导入 message 类型

LangChain 的 message 不是字符串拼接，而是有类型的对象。

先看最常见的四种：

- `SystemMessage`
- `HumanMessage`
- `AIMessage`
- `ToolMessage`

In [2]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

system_msg = SystemMessage(content="你是一个面向 Java 开发者的中文教学助手。")
user_msg = HumanMessage(content="请解释 LangChain 里的 messages 是什么。")
assistant_msg = AIMessage(content="messages 是模型输入输出的结构化上下文。")
tool_msg = ToolMessage(content="tool result: found 3 files", tool_call_id="call_demo_001")

messages = [system_msg, user_msg, assistant_msg, tool_msg]
for idx, msg in enumerate(messages, start=1):
    print(idx, type(msg).__name__, "|", msg.type, "|", msg.content)

1 SystemMessage | system | 你是一个面向 Java 开发者的中文教学助手。
2 HumanMessage | human | 请解释 LangChain 里的 messages 是什么。
3 AIMessage | ai | messages 是模型输入输出的结构化上下文。
4 ToolMessage | tool | tool result: found 3 files


## 4. 把 message 看成结构，而不是拼接文本

最重要的是：模型不是只看最后一句话，它看的是整个 message list。

这也是为什么上下文管理必须分层：

- system 决定规则
- history 决定对话连续性
- tool message 决定工具观察
- assistant message 决定上轮结论

本仓库 Harness 的 `completed_steps` 就是把这些观察再结构化一次，避免纯文本上下文失控。

In [3]:
def message_to_dict(message):
    return {
        "type": getattr(message, "type", message.__class__.__name__.lower()),
        "content": getattr(message, "content", str(message)),
    }


print([message_to_dict(msg) for msg in messages])

[{'type': 'system', 'content': '你是一个面向 Java 开发者的中文教学助手。'}, {'type': 'human', 'content': '请解释 LangChain 里的 messages 是什么。'}, {'type': 'ai', 'content': 'messages 是模型输入输出的结构化上下文。'}, {'type': 'tool', 'content': 'tool result: found 3 files'}]


## 5. 对比本仓库的上下文结构

如果把 LangChain 的 messages 映射到当前仓库，可以这样理解：

| LangChain | 本仓库 |
|---|---|
| `messages` | `history_snapshot` + `completed_steps` + 当前 `plan` |
| `SystemMessage` | planner/answer prompt |
| `HumanMessage` | 用户输入 |
| `AIMessage` | 上一轮模型输出 |
| `ToolMessage` | 工具执行结果 |

也就是说，LangChain 的消息模型本身并不神奇，关键还是你如何裁剪、归档和恢复这些上下文。

In [6]:
import json


def build_harness_style_messages(system_prompt: str, history_snapshot: list[dict[str, str]], user_message: str, completed_steps: list[dict] | None = None):
    payload = {
        "user_message": user_message,
        "completed_steps": completed_steps or [],
    }
    return [
        {"role": "system", "content": system_prompt},
        *history_snapshot,
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]


demo_history = [
    {"role": "user", "content": "帮我看一下 approval 恢复流程"},
    {"role": "assistant", "content": "我先检查 app/agents/harness.py"},
]
demo_completed_steps = [
    {
        "step_no": 1,
        "plan": {"action": "delegate", "role": "research"},
        "subagent_result": {"ok": True, "summary": "找到 resume_approval 和 stream_resume_approval"},
    }
]

print(build_harness_style_messages(
    system_prompt="你是一个 Harness 风格聊天智能体的规划器。",
    history_snapshot=demo_history,
    user_message="请继续分析 approval 恢复流程",
    completed_steps=demo_completed_steps,
))

[{'role': 'system', 'content': '你是一个 Harness 风格聊天智能体的规划器。'}, {'role': 'user', 'content': '帮我看一下 approval 恢复流程'}, {'role': 'assistant', 'content': '我先检查 app/agents/harness.py'}, {'role': 'user', 'content': '{"user_message": "请继续分析 approval 恢复流程", "completed_steps": [{"step_no": 1, "plan": {"action": "delegate", "role": "research"}, "subagent_result": {"ok": true, "summary": "找到 resume_approval 和 stream_resume_approval"}}]}'}]


## 6. 可选：创建一个真实模型对象

如果你已经配置好 `OPENAI_API_KEY`，可以创建 `ChatOpenAI`。

这一讲先不强依赖真实调用，重点是看懂 messages 结构。

In [7]:
from langchain_openai import ChatOpenAI

if not OPENAI_API_KEY:
    chat_model = None
    print("Skip ChatOpenAI because OPENAI_API_KEY is missing.")
else:
    chat_model = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )
    print("ChatOpenAI ready:", OPENAI_MODEL)

ChatOpenAI ready: qwq


## 7. 可选：真正发起一次模型调用

如果模型可用，就用前面定义的 messages 发起一次最小调用。

这一步不是必须的；如果没有 API key，Notebook 依然可以完整学完 message 结构。

In [8]:
if chat_model is None:
    print("Skip live invoke.")
else:
    response = chat_model.invoke([
        system_msg,
        HumanMessage(content="请用一句话解释 messages 和 history_snapshot 的差别。"),
    ])
    print(type(response).__name__)
    print(response.content)


AIMessage


`messages` 存储的是对话的原始逐条明细，而 `history_snapshot` 是特定时刻对话状态的压缩快照，前者侧重完整记录与追溯，后者侧重高效恢复上下文。


## 8. 本讲小结

这一讲只记住三件事：

1. LangChain 的核心输入是 message list，不是散乱文本。
2. `system / user / assistant / tool` 是上下文结构，不只是字符串标签。
3. 本仓库的 Harness runtime 本质上也是在管理 messages 的生命周期，只是它把控制流、ledger 和恢复显式化了。

下一讲适合继续学：

- LangChain tools 定义
- tool calling 的执行链路
- 如何把本仓库的工具系统映射过去